# iSTAMP revision analysis — 00 · Environment & data overview

This notebook series reproduces the **leakage-corrected** reanalysis of the iSTAMP
manuscript (*Intraoperative classification of glioblastoma through near real-time
stimulated Raman scattering microscopy*).

**Order:** `00` setup → `01` master inference → `02` leakage-corrected training →
`03` graph density & saturation → `04` external validation & MGMT → `05` figures.

**Environment:** create it with `conda env create -f ../environment.yml` (env name **`Raman_Analysis`**), then `conda activate Raman_Analysis` and register the
kernel with `python -m ipykernel install --user --name Raman_Analysis`.

> All model outputs are exact re-runs of the trained network — nothing is fabricated.

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)


## 1 · Verify the Python environment

In [ ]:
import torch, torch_geometric, numpy, pandas, sklearn, matplotlib
print('torch          ', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('numpy          ', numpy.__version__)
print('pandas         ', pandas.__version__)
print('scikit-learn   ', sklearn.__version__)
print('CUDA available ', torch.cuda.is_available(), '| MPS', torch.backends.mps.is_available())

## 2 · Cohort overview
`meta.csv` holds one row per SRH image (1,295 images, 236 patients, 4 centres).

In [ ]:
import pandas as pd
meta = pd.read_csv(L.META_PATH)
print('images  :', len(meta))
print('patients:', meta['Sample.Number.internal'].nunique())
print('centres :', meta['Klinik'].value_counts().to_dict())
print('subtypes:', meta['thisClass'].value_counts().to_dict())
print('MGMT    :', meta['MGMTStatus'].value_counts().to_dict())
meta[['image_id','Sample.Number.internal','Klinik','thisClass','MGMTStatus']].head()

## 3 · Load the trained model
`RamanGATv2MultiBias` (GATv2, in=512 CONCH features, hidden=256, 8 heads, 4 layers, 5 outputs, domain-adversarial heads). The checkpoint loads with **0 missing keys**.

In [ ]:
model, ckpt, missing, unexpected = L.load_model(device='cpu')
print('missing keys   :', len(missing))
print('unexpected keys:', len(unexpected))
print('hyperparameters:')
for k, v in ckpt['params'].items():
    print(f'  {k:24s} {v}')

## 4 · Single-graph inference demo
Each `Img_*_pyg.pt` file stores ego-graphs at several hops. We run one and compare
the predicted subtype probabilities with the methylation ground-truth target.

In [ ]:
import glob, numpy as np
f = sorted(glob.glob(L.DATA + '/Img_*_pyg.pt'))[0]
d = L.load_graph_dict(f)
g = d['pyg_dataset_by_hop'][3][0]           # hop-3 subgraph
pred, n_nodes = L.infer_graph(model, g)
probs = L.softmax(pred)
print('image', d['image_id'], '| patient', d['patient_id'], '| nodes', n_nodes)
for c, p in zip(L.CLASSES, probs):
    print(f'  {c:14s} {p:.3f}')
print('predicted :', L.CLASSES[int(probs.argmax())])
print('true label:', L.CLASSES[int(np.argmax(np.asarray(g.y).reshape(-1)[:5]))])